In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("dark_background")

# -----------------------------
# 1) Function f(x, y): two nearby Gaussians
# -----------------------------
def f(x, y):
    A1, x1, y1, s1 = 1.0,  1.0, 0.0, 0.7
    A2, x2, y2, s2 = 0.5, -1.0, 0.3, 0.4
    g1 = A1 * np.exp(-((x - x1)**2 + (y - y1)**2) / (2 * s1**2))
    g2 = A2 * np.exp(-((x - x2)**2 + (y - y2)**2) / (2 * s2**2))
    return g1 + g2

# -----------------------------------------
# 2) Numerical Hessian
# -----------------------------------------
def numerical_hessian(f, x0, y0, h=1e-4):
    f00 = f(x0, y0)

    f_xx = (f(x0 + h, y0) - 2 * f00 + f(x0 - h, y0)) / h**2
    f_yy = (f(x0, y0 + h) - 2 * f00 + f(x0, y0 - h)) / h**2

    f_xy = (
        f(x0 + h, y0 + h)
        - f(x0 + h, y0 - h)
        - f(x0 - h, y0 + h)
        + f(x0 - h, y0 - h)
    ) / (4 * h**2)

    return np.array([[f_xx, f_xy], [f_xy, f_yy]], dtype=float)

def hessian_eigs(f, x0, y0, h=1e-4):
    H = numerical_hessian(f, x0, y0, h=h)
    evals, evecs = np.linalg.eigh(H)  # symmetric => real eigs
    return evals, evecs, H

# -----------------------------
# 3) Precompute contour grid
# -----------------------------
x_min, x_max = -3.0, 3.0
y_min, y_max = -3.0, 3.0
n_grid = 240

xs = np.linspace(x_min, x_max, n_grid)
ys = np.linspace(y_min, y_max, n_grid)
X, Y = np.meshgrid(xs, ys)
Z = f(X, Y)

cmap = plt.cm.viridis
norm = plt.Normalize(Z.min(), Z.max())

# -----------------------------
# 4) Interactive contour (click to move point)
# -----------------------------
%matplotlib qt

# initial point
x0, y0 = 0.8, -0.6

# display settings
levels = 10
vec_len = 0.9          # arrow half-length in data units
h_step = 1e-4          # finite difference step for Hessian
arrow_lw = 4.0
text_fs = 16

fig, ax = plt.subplots(figsize=(8, 6))

cs = ax.contour(X, Y, Z, levels=levels, cmap=cmap, norm=norm)
ax.clabel(cs, inline=True, fontsize=8)

# Center axes spines at (0,0)
ax.spines["left"].set_position("zero")
ax.spines["bottom"].set_position("zero")
ax.spines["right"].set_color("none")
ax.spines["top"].set_color("none")
ax.xaxis.set_ticks_position("bottom")
ax.yaxis.set_ticks_position("left")

ax.set_aspect("equal", adjustable="box")
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.grid(True, alpha=0.3)
ax.set_title("Click to move (x0,y0): Hessian eigendirections & eigenvalues")

# Point artist
pt = ax.scatter([x0], [y0], color="white", s=70, zorder=6)

# Eigenvector artists (as thick red line segments)
line1, = ax.plot([], [], color="red", linewidth=arrow_lw, zorder=7)
line2, = ax.plot([], [], color="red", linewidth=arrow_lw, zorder=7)

# Eigenvalue text
txt1 = ax.text(0, 0, "", color="red", fontsize=text_fs, weight="bold", zorder=8)
txt2 = ax.text(0, 0, "", color="red", fontsize=text_fs, weight="bold", zorder=8)

# Small status box
status = ax.text(
    0.02, 0.02, "",
    transform=ax.transAxes,
    color="white",
    fontsize=11,
    bbox=dict(facecolor="black", alpha=0.45, edgecolor="white", linewidth=0.5, pad=6),
    zorder=9
)

def set_eigs_artists(x0, y0):
    evals, evecs, H = hessian_eigs(f, x0, y0, h=h_step)
    v1 = evecs[:, 0]
    v2 = evecs[:, 1]

    # segments centered at (x0,y0): from -v to +v
    x1 = [x0 - vec_len*v1[0], x0 + vec_len*v1[0]]
    y1 = [y0 - vec_len*v1[1], y0 + vec_len*v1[1]]
    x2 = [x0 - vec_len*v2[0], x0 + vec_len*v2[0]]
    y2 = [y0 - vec_len*v2[1], y0 + vec_len*v2*v2[1]]  # <-- typo guard below

    # typo guard (in case of copy edits)
    if isinstance(y2[1], np.ndarray) or isinstance(y2[1], list):
        y2 = [y0 - vec_len*v2[1], y0 + vec_len*v2[1]]

    line1.set_data(x1, y1)
    line2.set_data(x2, y2)

    # place labels near one endpoint of each line
    txt1.set_position((x1[1], y1[1]))
    txt1.set_text(f"λ={evals[0]:.3g}")

    txt2.set_position((x2[1], y2[1]))
    txt2.set_text(f"λ={evals[1]:.3g}")

    status.set_text(
        f"(x0,y0)=({x0:.3f},{y0:.3f})\n"
        f"λ=[{evals[0]:.6g}, {evals[1]:.6g}]\n"
        f"H=\n[{H[0,0]: .3g}  {H[0,1]: .3g}]\n"
        f"  [{H[1,0]: .3g}  {H[1,1]: .3g}]"
    )

def update_all(x0, y0):
    pt.set_offsets([[x0, y0]])
    set_eigs_artists(x0, y0)
    fig.canvas.draw_idle()

# Initial draw
update_all(x0, y0)
plt.tight_layout()
plt.show()

def on_click(event):
    # Only respond to clicks inside the axes
    if event.inaxes != ax:
        return
    if event.xdata is None or event.ydata is None:
        return

    x0_new, y0_new = float(event.xdata), float(event.ydata)
    update_all(x0_new, y0_new)

cid = fig.canvas.mpl_connect("button_press_event", on_click)

print("Interactive mode: click anywhere in the contour plot to move (x0,y0).")
print("If you want to change vec_len or h_step, edit vec_len/h_step above and re-run the cell.")


Interactive mode: click anywhere in the contour plot to move (x0,y0).
If you want to change vec_len or h_step, edit vec_len/h_step above and re-run the cell.


In [4]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("dark_background")

# -----------------------------
# 1) Function f(x, y): two nearby Gaussians
# -----------------------------
def f(x, y):
    A1, x1, y1, s1 = 1.0,  1.0, 0.0, 0.7
    A2, x2, y2, s2 = 0.5, -1.0, 0.3, 0.4
    g1 = A1 * np.exp(-((x - x1)**2 + (y - y1)**2) / (2 * s1**2))
    g2 = A2 * np.exp(-((x - x2)**2 + (y - y2)**2) / (2 * s2**2))
    return g1 + g2

# -----------------------------------------
# 2) Numerical derivatives: gradient + Hessian
# -----------------------------------------
def numerical_gradient(f, x0, y0, h=1e-5):
    fx = (f(x0 + h, y0) - f(x0 - h, y0)) / (2 * h)
    fy = (f(x0, y0 + h) - f(x0, y0 - h)) / (2 * h)
    return np.array([fx, fy], dtype=float)

def numerical_hessian(f, x0, y0, h=1e-4):
    f00 = f(x0, y0)

    f_xx = (f(x0 + h, y0) - 2 * f00 + f(x0 - h, y0)) / h**2
    f_yy = (f(x0, y0 + h) - 2 * f00 + f(x0, y0 - h)) / h**2

    f_xy = (
        f(x0 + h, y0 + h)
        - f(x0 + h, y0 - h)
        - f(x0 - h, y0 + h)
        + f(x0 - h, y0 - h)
    ) / (4 * h**2)

    return np.array([[f_xx, f_xy], [f_xy, f_yy]], dtype=float)

def hessian_eigs(f, x0, y0, h=1e-4):
    H = numerical_hessian(f, x0, y0, h=h)
    evals, evecs = np.linalg.eigh(H)  # symmetric => real eigs
    return evals, evecs, H

# -----------------------------
# 3) Precompute contour grid
# -----------------------------
x_min, x_max = -3.0, 3.0
y_min, y_max = -3.0, 3.0
n_grid = 240

xs = np.linspace(x_min, x_max, n_grid)
ys = np.linspace(y_min, y_max, n_grid)
X, Y = np.meshgrid(xs, ys)
Z = f(X, Y)

cmap = plt.cm.viridis
norm = plt.Normalize(Z.min(), Z.max())

# -----------------------------
# 4) Interactive contour (click to move point)
# -----------------------------
%matplotlib qt

# initial point
x0, y0 = 0.8, -0.6

# display settings
levels = 10
h_step_hess = 1e-4      # finite difference step for Hessian
h_step_grad = 1e-5      # finite difference step for gradient
base_len = 0.55         # overall visual scale for eigenvector arrows
eig_width = 0.008
grad_width = 0.008

# gradient arrow settings (visual)
grad_scale = 0.9

fig, ax = plt.subplots(figsize=(8, 6))

cs = ax.contour(X, Y, Z, levels=levels, cmap=cmap, norm=norm)
ax.clabel(cs, inline=True, fontsize=8)

# Center axes spines at (0,0)
ax.spines["left"].set_position("zero")
ax.spines["bottom"].set_position("zero")
ax.spines["right"].set_color("none")
ax.spines["top"].set_color("none")
ax.xaxis.set_ticks_position("bottom")
ax.yaxis.set_ticks_position("left")

ax.set_aspect("equal", adjustable="box")
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.grid(True, alpha=0.3)
ax.set_title("Click to move (x0,y0): Hessian eigendirections as quivers scaled by sqrt(|λ|) + gradient")

# Point artist
pt = ax.scatter([x0], [y0], color="white", s=70, zorder=6)

# Eigenvector quivers (draw both + and - directions, scaled by sqrt(|λ|))
eig_q = ax.quiver(
    [x0, x0, x0, x0], [y0, y0, y0, y0],
    [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0],
    angles="xy", scale_units="xy", scale=1.0,
    width=eig_width, color="red", zorder=7
)

# Gradient quiver (single arrow)
grad_q = ax.quiver(
    [x0], [y0], [0.0], [0.0],
    angles="xy", scale_units="xy", scale=1.0,
    width=grad_width, color="white", zorder=8
)

# Small status box
status = ax.text(
    0.02, 0.02, "",
    transform=ax.transAxes,
    color="white",
    fontsize=11,
    bbox=dict(facecolor="black", alpha=0.45, edgecolor="white", linewidth=0.5, pad=6),
    zorder=9
)

def _safe_sqrt_scale(lam, eps=1e-12):
    return np.sqrt(max(abs(float(lam)), eps))

def set_artists(x0, y0):
    evals, evecs, H = hessian_eigs(f, x0, y0, h=h_step_hess)

    # eigh returns ascending; keep that order, just display both
    v1 = evecs[:, 0]
    v2 = evecs[:, 1]

    # scale eigenvectors by sqrt(|lambda|)
    s1 = base_len * _safe_sqrt_scale(evals[0])
    s2 = base_len * _safe_sqrt_scale(evals[1])

    # Build 4 arrows: +v1, -v1, +v2, -v2
    U = np.array([ s1 * v1[0], -s1 * v1[0],  s2 * v2[0], -s2 * v2[0]], dtype=float)
    V = np.array([ s1 * v1[1], -s1 * v1[1],  s2 * v2[1], -s2 * v2[1]], dtype=float)

    # Update eigen quiver positions and directions
    eig_q.set_offsets(np.array([[x0, y0],
                                [x0, y0],
                                [x0, y0],
                                [x0, y0]], dtype=float))
    eig_q.set_UVC(U, V)

    # Gradient at point
    g = numerical_gradient(f, x0, y0, h=h_step_grad)
    grad_q.set_offsets(np.array([[x0, y0]], dtype=float))
    grad_q.set_UVC(np.array([grad_scale * g[0]]), np.array([grad_scale * g[1]]))

    status.set_text(
        f"(x0,y0)=({x0:.3f},{y0:.3f})\n"
        f"grad=[{g[0]:.6g}, {g[1]:.6g}]  |grad|={np.linalg.norm(g):.3g}\n"
        f"λ=[{evals[0]:.6g}, {evals[1]:.6g}]\n"
        f"H=\n[{H[0,0]: .3g}  {H[0,1]: .3g}]\n"
        f"  [{H[1,0]: .3g}  {H[1,1]: .3g}]"
    )

def update_all(x0, y0):
    pt.set_offsets([[x0, y0]])
    set_artists(x0, y0)
    fig.canvas.draw_idle()

# Initial draw
update_all(x0, y0)
plt.tight_layout()
plt.show()

def on_click(event):
    if event.inaxes != ax:
        return
    if event.xdata is None or event.ydata is None:
        return
    update_all(float(event.xdata), float(event.ydata))

cid = fig.canvas.mpl_connect("button_press_event", on_click)

print("Interactive mode: click anywhere in the contour plot to move (x0,y0).")
print("Eigenvectors are drawn as quivers in both + and - directions, scaled by sqrt(|λ|).")
print("To change eigenvector visibility, edit base_len or eig_width and re-run the cell.")
print("To change gradient visibility, edit grad_scale or grad_width and re-run the cell.")
print("To change derivative steps, edit h_step_hess / h_step_grad above and re-run the cell.")

Interactive mode: click anywhere in the contour plot to move (x0,y0).
Eigenvectors are drawn as quivers in both + and - directions, scaled by sqrt(|λ|).
To change eigenvector visibility, edit base_len or eig_width and re-run the cell.
To change gradient visibility, edit grad_scale or grad_width and re-run the cell.
To change derivative steps, edit h_step_hess / h_step_grad above and re-run the cell.
